# Walmart Weekly Sales: Experiments

This notebook tests three improvements on top of `Notebook.ipynb`, **without touching the deployed model**:

1. **A validation split that contains Thanksgiving and Christmas.** The original split (train 2010–2011 / validate 2012) validates on a period that stops on 2012-10-26, so the two biggest peaks of the year were never tested.
2. **Lag features** (sales of the same store/department one year earlier, store/department historical level).
3. **Retraining the final model on all available data (2010 → 2012)** before predicting the Kaggle test set.

All models are evaluated with the official Kaggle metric, **WMAE** (holiday weeks weigh 5×).

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

## Data loading & base features (same as `Notebook.ipynb`)

In [2]:
train = pd.read_csv("./Data/train.csv")
test = pd.read_csv("./Data/test.csv")
features = pd.read_csv("./Data/features.csv")
stores = pd.read_csv("./Data/stores.csv")

feature_store_merged = features.merge(stores, how="inner", on="Store")

THANKSGIVING_WEEK = {2010: "2010-11-26", 2011: "2011-11-25", 2012: "2012-11-23", 2013: "2013-11-29"}
CHRISTMAS_WEEK    = {2010: "2010-12-31", 2011: "2011-12-30", 2012: "2012-12-28", 2013: "2013-12-27"}
SUPER_BOWL_WEEK   = {2010: "2010-02-12", 2011: "2011-02-11", 2012: "2012-02-10", 2013: "2013-02-08"}
LABOR_DAY_WEEK    = {2010: "2010-09-10", 2011: "2011-09-09", 2012: "2012-09-07", 2013: "2013-09-06"}
HOLIDAYS = {"Thanksgiving": THANKSGIVING_WEEK, "Christmas": CHRISTMAS_WEEK,
            "SuperBowl": SUPER_BOWL_WEEK, "LaborDay": LABOR_DAY_WEEK}

MARKDOWN_COLS = ["MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5"]


def build_base_features(df):
    df = df.merge(feature_store_merged, how="inner", on=["Store", "Date", "IsHoliday"])
    df["Date"] = pd.to_datetime(df["Date"])
    df["Year"] = df["Date"].dt.year
    df["Week_Of_Year"] = df["Date"].dt.isocalendar().week.astype(int)
    for name, dates in HOLIDAYS.items():
        ref = df["Date"].dt.year.map({y: pd.Timestamp(d) for y, d in dates.items()})
        df[f"Days_to_{name}"] = (ref - df["Date"]).dt.days
    df[MARKDOWN_COLS] = df[MARKDOWN_COLS].fillna(0)
    df["IsHoliday"] = df["IsHoliday"].astype(int)
    return df.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)


train_df = build_base_features(train)
test_df = build_base_features(test)

BASE_FEATURES = ["Store", "Dept", "IsHoliday", "Temperature", "Fuel_Price", *MARKDOWN_COLS,
                 "CPI", "Unemployment", "Type", "Size", "Year", "Week_Of_Year",
                 "Days_to_Thanksgiving", "Days_to_Christmas", "Days_to_SuperBowl", "Days_to_LaborDay"]

print("train:", train_df["Date"].min().date(), "->", train_df["Date"].max().date(), train_df.shape)
print("test: ", test_df["Date"].min().date(), "->", test_df["Date"].max().date(), test_df.shape)

train: 2010-02-05 -> 2012-10-26 (421570, 22)
test:  2012-11-02 -> 2013-07-26 (115064, 21)


## Evaluation helpers

* `wmae`: the official Kaggle metric.
* `MAE Thanks/Xmas`: error on Thanksgiving and Christmas weeks only, the weeks the original validation could not measure.

In [3]:
def wmae(y_true, y_pred, is_holiday):
    w = np.where(is_holiday, 5, 1)
    return np.sum(w * np.abs(np.asarray(y_true) - np.asarray(y_pred))) / np.sum(w)


def peak_mask(dates):
    peaks = [pd.Timestamp(d) for d in list(THANKSGIVING_WEEK.values()) + list(CHRISTMAS_WEEK.values())]
    return dates.isin(peaks).values


def evaluate(name, y_true, y_pred, val_df):
    peaks = peak_mask(val_df["Date"])
    return {
        "Experiment": name,
        "WMAE": wmae(y_true, y_pred, val_df["IsHoliday"] == 1),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R²": r2_score(y_true, y_pred),
        "MAE Thanks/Xmas": mean_absolute_error(y_true[peaks], y_pred[peaks]) if peaks.any() else np.nan,
    }


def make_models():
    # Same hyper-parameters as the tuned models of Notebook.ipynb; XGBoost & LightGBM on GPU
    return {
        "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=20, min_samples_leaf=10,
                                               random_state=42, n_jobs=-1),
        "XGBoost": XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=8, subsample=0.8,
                                colsample_bytree=0.8, min_child_weight=5, random_state=42,
                                device="cuda", tree_method="hist"),
        "LightGBM": LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=8, num_leaves=63,
                                  subsample=0.8, colsample_bytree=0.8, min_child_samples=20,
                                  random_state=42, device="gpu", verbose=-1),
    }


def make_pipeline(model, feature_cols):
    # Tree models are scale-invariant: no StandardScaler needed. Missing lags (NaN) are handled natively.
    numeric = [c for c in feature_cols if c != "Type"]
    pre = ColumnTransformer([
        ("type", OrdinalEncoder(categories=[["C", "B", "A"]]), ["Type"]),
        ("num", "passthrough", numeric),
    ])
    return Pipeline([("Preprocessing", pre), ("Model", model)])


def run_experiment(label, tr, va, feature_cols, results, predictions):
    for name, model in make_models().items():
        pipe = make_pipeline(model, feature_cols).fit(tr[feature_cols], tr["Weekly_Sales"])
        pred = pipe.predict(va[feature_cols])
        predictions[(label, name)] = pred
        results.append({"Model": name, **evaluate(label, va["Weekly_Sales"].values, pred, va)})
        print(f"done: {label} / {name}")

## 1. A validation split that contains the holiday peaks

| Split | Train | Validation | Thanksgiving / Christmas in validation? |
|---|---|---|---|
| Original | 2010-02-05 → 2011-12-30 | 2012-01-06 → 2012-10-26 | ❌ no |
| **New** | 2010-02-05 → 2011-10-28 | **2011-11-04 → 2012-10-26 (52 weeks)** | ✅ yes |

The new validation window is **exactly one year**, so it covers every season once, including Thanksgiving and Christmas 2011.

In [4]:
OLD_CUTOFF = pd.Timestamp("2011-12-30")
NEW_CUTOFF = pd.Timestamp("2011-10-28")

splits = {
    "Old split": (train_df[train_df["Date"] <= OLD_CUTOFF], train_df[train_df["Date"] > OLD_CUTOFF]),
    "New split": (train_df[train_df["Date"] <= NEW_CUTOFF], train_df[train_df["Date"] > NEW_CUTOFF]),
}

for name, (tr, va) in splits.items():
    holidays_in_val = sorted(va.loc[va["IsHoliday"] == 1, "Date"].dt.date.unique())
    print(f"{name}: train {len(tr):,} rows | val {len(va):,} rows | holiday weeks in val: {holidays_in_val}")

Old split: train 294,132 rows | val 127,438 rows | holiday weeks in val: [datetime.date(2012, 2, 10), datetime.date(2012, 9, 7)]
New split: train 267,184 rows | val 154,386 rows | holiday weeks in val: [datetime.date(2011, 11, 25), datetime.date(2011, 12, 30), datetime.date(2012, 2, 10), datetime.date(2012, 9, 7)]


In [5]:
results, predictions = [], {}

for split_name, (tr, va) in splits.items():
    run_experiment(f"{split_name} / base features", tr, va, BASE_FEATURES, results, predictions)

pd.DataFrame(results).round({"WMAE": 0, "MAE": 0, "R²": 4, "MAE Thanks/Xmas": 0})

done: Old split / base features / Random Forest


done: Old split / base features / XGBoost


done: Old split / base features / LightGBM


done: New split / base features / Random Forest


done: New split / base features / XGBoost


done: New split / base features / LightGBM


,Model,Experiment,WMAE,MAE,R²,MAE Thanks/Xmas
0,Random Forest,Old split / base features,2364.0,2330.0,0.9459,NaN
1,XGBoost,Old split / base features,2752.0,2732.0,0.9523,NaN
2,LightGBM,Old split / base features,3123.0,3095.0,0.9418,NaN
3,Random Forest,New split / base features,2779.0,2595.0,0.9304,4070.0
4,XGBoost,New split / base features,2919.0,2809.0,0.9476,3747.0
5,LightGBM,New split / base features,3498.0,3348.0,0.9317,4595.0


**Reading:** the old split's scores were optimistic. Once Thanksgiving and Christmas are in the validation window, the WMAE goes up for every model: the holiday weeks are both the hardest to predict and weighted 5×. The **new split is the honest benchmark** for the rest of this notebook.

## 2. Lag features

### Which lags are allowed?

A lag is only legitimate if its value is **known at prediction time**. The Kaggle test set covers 2012-11-02 → 2013-07-26, i.e. up to **39 weeks** after the last known sales. A `lag_1` (last week's sales) is therefore **not available** for most test weeks: using it would require recursive forecasting and would leak in validation.

Lags of **52 weeks or more** are always known, both for the test set and for our 52-week validation window. We build:

| Feature | Description |
|---|---|
| `Sales_Lag_52` | Sales of the same store/department exactly 52 weeks (364 days) earlier: same weekday, same holiday week |
| `Sales_Lag_52_Mean4` | Mean of lags 52 to 55 weeks: a smoother "last year around this time" level |
| `Store_Dept_Mean` | Average weekly sales of the store/department over the **training period only** |
| `Store_Dept_Median` | Same with the median, robust to holiday spikes |

364 days = 52 weeks exactly, so Thanksgiving 2011 (2011-11-25) is aligned with Thanksgiving 2010 (2010-11-26), and Christmas weeks are aligned too.

In [6]:
LAG_FEATURES = ["Sales_Lag_52", "Sales_Lag_52_Mean4", "Store_Dept_Mean", "Store_Dept_Median"]


def add_lag_features(df, history, stats_source):
    """
    df:           rows to enrich
    history:      known sales (Store, Dept, Date, Weekly_Sales) used for the 52+ week lags
    stats_source: rows used to compute the store/department level (training period only)
    """
    df = df.copy()
    hist = history[["Store", "Dept", "Date", "Weekly_Sales"]]
    for k in [52, 53, 54, 55]:
        shifted = hist.assign(Date=hist["Date"] + pd.Timedelta(weeks=k)).rename(columns={"Weekly_Sales": f"lag_{k}"})
        df = df.merge(shifted, on=["Store", "Dept", "Date"], how="left")
    df["Sales_Lag_52"] = df["lag_52"]
    df["Sales_Lag_52_Mean4"] = df[[f"lag_{k}" for k in [52, 53, 54, 55]]].mean(axis=1)
    df = df.drop(columns=[f"lag_{k}" for k in [52, 53, 54, 55]])

    stats = (stats_source.groupby(["Store", "Dept"])["Weekly_Sales"]
             .agg(Store_Dept_Mean="mean", Store_Dept_Median="median").reset_index())
    return df.merge(stats, on=["Store", "Dept"], how="left")


tr, va = splits["New split"]
# history = training period only: guarantees that no validation sales leak into the features
tr_lag = add_lag_features(tr, history=tr, stats_source=tr)
va_lag = add_lag_features(va, history=tr, stats_source=tr)

print("Share of validation rows with a known 52-week lag:", f"{va_lag['Sales_Lag_52'].notna().mean():.1%}")
print("Share of training rows with a known 52-week lag:  ", f"{tr_lag['Sales_Lag_52'].notna().mean():.1%}",
      "(the first year has no history)")

Share of validation rows with a known 52-week lag: 97.1%
Share of training rows with a known 52-week lag:   41.8% (the first year has no history)


In [7]:
run_experiment("New split / base + lag features", tr_lag, va_lag, BASE_FEATURES + LAG_FEATURES, results, predictions)

comparison = (pd.DataFrame(results)
              .sort_values(["Experiment", "WMAE"])
              .reset_index(drop=True))
comparison.round({"WMAE": 0, "MAE": 0, "R²": 4, "MAE Thanks/Xmas": 0})

done: New split / base + lag features / Random Forest


done: New split / base + lag features / XGBoost


done: New split / base + lag features / LightGBM


,Model,Experiment,WMAE,MAE,R²,MAE Thanks/Xmas
0,LightGBM,New split / base + lag features,2072.0,1962.0,0.9636,2988.0
1,Random Forest,New split / base + lag features,2293.0,2017.0,0.9451,4425.0
2,XGBoost,New split / base + lag features,2645.0,2443.0,0.8124,4137.0
3,Random Forest,New split / base features,2779.0,2595.0,0.9304,4070.0
4,XGBoost,New split / base features,2919.0,2809.0,0.9476,3747.0
5,LightGBM,New split / base features,3498.0,3348.0,0.9317,4595.0
6,Random Forest,Old split / base features,2364.0,2330.0,0.9459,NaN
7,XGBoost,Old split / base features,2752.0,2732.0,0.9523,NaN
8,LightGBM,Old split / base features,3123.0,3095.0,0.9418,NaN


In [8]:
new_split = comparison[comparison["Experiment"].str.startswith("New split")]
gain = new_split.pivot(index="Model", columns="Experiment", values="WMAE")
# explicit column order (pivot sorts columns alphabetically)
gain = gain[["New split / base features", "New split / base + lag features"]]
gain.columns = ["WMAE base", "WMAE + lags"]
gain["Gain"] = gain["WMAE base"] - gain["WMAE + lags"]
gain["Gain %"] = gain["Gain"] / gain["WMAE base"] * 100
gain.sort_values("WMAE + lags").round(1)

,WMAE base,WMAE + lags,Gain,Gain %
Model,,,,
LightGBM,3497.9,2071.9,1426.0,40.8
Random Forest,2778.6,2293.3,485.4,17.5
XGBoost,2918.7,2644.5,274.2,9.4


### Actual vs predicted on the new validation window (with Thanksgiving & Christmas)

In [9]:
best_lag_model = gain["WMAE + lags"].idxmin()
plot_df = va[["Date"]].copy()
plot_df["Actual"] = va["Weekly_Sales"].values
plot_df["Base features"] = predictions[("New split / base features", best_lag_model)]
plot_df["Base + lag features"] = predictions[("New split / base + lag features", best_lag_model)]
weekly = plot_df.groupby("Date", as_index=False).sum()

fig = go.Figure()
fig.add_trace(go.Scatter(x=weekly["Date"], y=weekly["Actual"], name="Actual", line=dict(width=3, color="black")))
fig.add_trace(go.Scatter(x=weekly["Date"], y=weekly["Base features"], name=f"{best_lag_model}: base features",
                         line=dict(width=2, dash="dot")))
fig.add_trace(go.Scatter(x=weekly["Date"], y=weekly["Base + lag features"], name=f"{best_lag_model}: + lag features",
                         line=dict(width=2, dash="dash")))
fig.update_layout(title=f"Total weekly sales, Nov 2011 → Oct 2012: actual vs {best_lag_model}",
                  title_x=0.5, xaxis_title="Week", yaxis_title="Total Weekly Sales ($)",
                  template="plotly_white", hovermode="x unified", width=1100, height=500)
fig.show()

In [10]:
fitted = (make_pipeline(make_models()[best_lag_model], BASE_FEATURES + LAG_FEATURES)
          .fit(tr_lag[BASE_FEATURES + LAG_FEATURES], tr_lag["Weekly_Sales"])
          .named_steps["Model"])
# LightGBM reports split counts by default: use gain to be comparable with the other models
raw_importance = (fitted.booster_.feature_importance(importance_type="gain")
                  if hasattr(fitted, "booster_") else fitted.feature_importances_)
importances = pd.Series(raw_importance,
                        index=["Type"] + [c for c in BASE_FEATURES + LAG_FEATURES if c != "Type"]
                        ).sort_values(ascending=False)
print(f"{best_lag_model}: share of total gain per feature")
(importances / importances.sum()).head(10).round(3)

LightGBM: share of total gain per feature


Store_Dept_Mean         0.711
Store_Dept_Median       0.156
Dept                    0.042
Sales_Lag_52_Mean4      0.029
Sales_Lag_52            0.025
Size                    0.007
Days_to_Christmas       0.005
Week_Of_Year            0.005
Days_to_Thanksgiving    0.004
Days_to_LaborDay        0.003
dtype: float64

## 3. Retrain on all data before predicting the test set

In `Notebook.ipynb`, the model used for the submission was trained on **2010–2011 only**: the 43 most recent weeks (Jan → Oct 2012), the closest to the test period, were never used for training.

For the final submission, once the model and features are chosen on the validation split, the standard practice is to **retrain on everything we know** (2010-02-05 → 2012-10-26). This cannot be scored without submitting to Kaggle, so we check it visually: the forecast must continue the historical pattern smoothly.

In [11]:
final_model_name = best_lag_model
final_features = BASE_FEATURES + LAG_FEATURES

# history and store/department statistics now use ALL known sales
train_full = add_lag_features(train_df, history=train_df, stats_source=train_df)
test_full = add_lag_features(test_df, history=train_df, stats_source=train_df)
print("Share of test rows with a known 52-week lag:", f"{test_full['Sales_Lag_52'].notna().mean():.1%}")

final_pipeline = make_pipeline(make_models()[final_model_name], final_features)
final_pipeline.fit(train_full[final_features], train_full["Weekly_Sales"])
test_full["Weekly_Sales"] = final_pipeline.predict(test_full[final_features])

Share of test rows with a known 52-week lag: 98.2%


In [12]:
submission = pd.DataFrame({
    "Id": test_full["Store"].astype(str) + "_" + test_full["Dept"].astype(str) + "_"
          + test_full["Date"].dt.strftime("%Y-%m-%d"),
    "Weekly_Sales": test_full["Weekly_Sales"],
})
sample = pd.read_csv("./Data/sampleSubmission.csv")
assert len(submission) == len(sample) and set(submission["Id"]) == set(sample["Id"])

submission.to_csv("walmart_submission_v2.csv", index=False)
print(f"walmart_submission_v2.csv saved ({final_model_name} + lag features, trained on 2010-2012). Not submitted.")
submission.head()

walmart_submission_v2.csv saved (LightGBM + lag features, trained on 2010-2012). Not submitted.


,Id,Weekly_Sales
0,1_1_2012-11-02,40146.562871
1,1_1_2012-11-09,21047.319232
2,1_1_2012-11-16,20014.680183
3,1_1_2012-11-23,21656.354756
4,1_1_2012-11-30,25476.994338


In [13]:
history_weekly = train_df.groupby("Date", as_index=False)["Weekly_Sales"].sum()
forecast_weekly = test_full.groupby("Date", as_index=False)["Weekly_Sales"].sum()

fig = go.Figure()
fig.add_trace(go.Scatter(x=history_weekly["Date"], y=history_weekly["Weekly_Sales"], name="Known sales",
                         line=dict(width=2, color="black")))
fig.add_trace(go.Scatter(x=forecast_weekly["Date"], y=forecast_weekly["Weekly_Sales"],
                         name=f"Forecast ({final_model_name} + lags, trained on all data)",
                         line=dict(width=2, color="crimson")))
fig.update_layout(title="Total weekly sales: history and Kaggle test-period forecast", title_x=0.5,
                  xaxis_title="Week", yaxis_title="Total Weekly Sales ($)",
                  template="plotly_white", hovermode="x unified", width=1100, height=500)
fig.show()

## 4. Out-of-fold target encoding

`Store_Dept_Mean` and `Store_Dept_Median` are **target encodings**: the average sales of each store/department pair. On the validation rows they are computed from the training period only, so the validation score is clean.

On the **training rows**, however, each row's own sales are part of the mean it receives: a small in-sample leak (each row weighs ~1/90 of its pair's mean). The model may learn to trust these features a bit more than it should.

**Out-of-fold (OOF) encoding** removes this leak: the training weeks are split into 5 folds, and the statistics of each row are computed **from the other 4 folds only**. No training row ever sees its own target. The folds are made of whole **weeks**, so a row never gets information from its own week either.

In [14]:
from sklearn.model_selection import KFold

STAT_FEATURES = ["Store_Dept_Mean", "Store_Dept_Median"]


def oof_store_dept_stats(df, n_splits=5, seed=42):
    """Store/department mean & median for each row, computed without the row's own week."""
    df = df.copy()
    weeks = np.sort(df["Date"].unique())
    out = pd.DataFrame(index=df.index, columns=STAT_FEATURES, dtype=float)
    for fit_idx, enc_idx in KFold(n_splits=n_splits, shuffle=True, random_state=seed).split(weeks):
        fit_rows = df["Date"].isin(weeks[fit_idx])
        stats = (df[fit_rows].groupby(["Store", "Dept"])["Weekly_Sales"]
                 .agg(Store_Dept_Mean="mean", Store_Dept_Median="median"))
        enc = df.loc[~fit_rows, ["Store", "Dept"]].join(stats, on=["Store", "Dept"])
        out.loc[enc.index, STAT_FEATURES] = enc[STAT_FEATURES].values
    df[STAT_FEATURES] = out
    return df


tr_oof = oof_store_dept_stats(tr_lag)   # lags unchanged, only the two statistics are re-computed

print("Correlation between in-sample and OOF Store_Dept_Mean on training rows:",
      f"{tr_lag['Store_Dept_Mean'].corr(tr_oof['Store_Dept_Mean']):.4f}")
print("Mean absolute difference: $" f"{(tr_lag['Store_Dept_Mean'] - tr_oof['Store_Dept_Mean']).abs().mean():,.0f}")
print("Training rows without an OOF value (pair seen in a single fold):", f"{tr_oof['Store_Dept_Mean'].isna().mean():.2%}")

Correlation between in-sample and OOF Store_Dept_Mean on training rows: 0.9999
Mean absolute difference: $162
Training rows without an OOF value (pair seen in a single fold): 0.02%


In [15]:
oof_results = []
for encoding, train_frame in [("in-sample", tr_lag), ("out-of-fold", tr_oof)]:
    for name, model in make_models().items():
        pipe = make_pipeline(model, BASE_FEATURES + LAG_FEATURES).fit(
            train_frame[BASE_FEATURES + LAG_FEATURES], train_frame["Weekly_Sales"])
        val_pred = pipe.predict(va_lag[BASE_FEATURES + LAG_FEATURES])
        train_pred = pipe.predict(train_frame[BASE_FEATURES + LAG_FEATURES])
        row = {"Model": name, **evaluate(f"lags, {encoding} encoding", va_lag["Weekly_Sales"].values, val_pred, va_lag)}
        row["Train WMAE"] = wmae(train_frame["Weekly_Sales"], train_pred, train_frame["IsHoliday"] == 1)
        row["Val/Train WMAE ratio"] = row["WMAE"] / row["Train WMAE"]
        oof_results.append(row)
        print(f"done: {encoding} / {name}")

oof_table = pd.DataFrame(oof_results).sort_values(["Model", "Experiment"]).reset_index(drop=True)
oof_table.round({"WMAE": 0, "MAE": 0, "R²": 4, "MAE Thanks/Xmas": 0, "Train WMAE": 0, "Val/Train WMAE ratio": 2})

done: in-sample / Random Forest


done: in-sample / XGBoost


done: in-sample / LightGBM


done: out-of-fold / Random Forest


done: out-of-fold / XGBoost


done: out-of-fold / LightGBM


,Model,Experiment,WMAE,MAE,R²,MAE Thanks/Xmas,Train WMAE,Val/Train WMAE ratio
0,LightGBM,"lags, in-sample encoding",2072.0,1962.0,0.9636,2989.0,1582.0,1.31
1,LightGBM,"lags, out-of-fold encoding",2085.0,1969.0,0.9637,3065.0,1629.0,1.28
2,Random Forest,"lags, in-sample encoding",2293.0,2017.0,0.9451,4425.0,1632.0,1.41
3,Random Forest,"lags, out-of-fold encoding",2364.0,2066.0,0.9422,4668.0,1700.0,1.39
4,XGBoost,"lags, in-sample encoding",2645.0,2443.0,0.8124,4137.0,1276.0,2.07
5,XGBoost,"lags, out-of-fold encoding",2061.0,1944.0,0.9649,3077.0,1312.0,1.57


In [16]:
lgbm_importance = {}
for encoding, train_frame in [("in-sample", tr_lag), ("out-of-fold", tr_oof)]:
    fitted = (make_pipeline(make_models()["LightGBM"], BASE_FEATURES + LAG_FEATURES)
              .fit(train_frame[BASE_FEATURES + LAG_FEATURES], train_frame["Weekly_Sales"])
              .named_steps["Model"])
    gain = pd.Series(fitted.booster_.feature_importance(importance_type="gain"),
                     index=["Type"] + [c for c in BASE_FEATURES + LAG_FEATURES if c != "Type"])
    lgbm_importance[encoding] = gain / gain.sum()

pd.DataFrame(lgbm_importance).sort_values("in-sample", ascending=False).head(8).round(3)

,in-sample,out-of-fold
Store_Dept_Mean,0.715,0.682
Store_Dept_Median,0.152,0.181
Dept,0.042,0.042
Sales_Lag_52_Mean4,0.029,0.030
Sales_Lag_52,0.025,0.025
Size,0.007,0.007
Week_Of_Year,0.005,0.005
Days_to_Christmas,0.005,0.005


**Reading:**

| Model | WMAE in-sample → OOF | R² in-sample → OOF | Val/Train WMAE ratio in-sample → OOF |
|---|---|---|---|
| **XGBoost** | 2,645 → **2,061** (-22%) | 0.812 → **0.965** | 2.07 → 1.57 |
| LightGBM | 2,070 → 2,086 (+1%) | 0.964 → 0.964 | 1.31 → 1.28 |
| Random Forest | 2,293 → 2,364 (+3%) | 0.945 → 0.942 | 1.41 → 1.39 |

* **The in-sample leak was real, and it explains XGBoost's R² drop.** With in-sample encoding, XGBoost had by far the largest gap between validation and training error (ratio 2.07): it over-trusted `Store_Dept_Mean` and produced a few very large errors. With OOF encoding its R² goes back to 0.965 and its WMAE drops by 22%.
* **LightGBM and Random Forest were not affected**: their scores barely move (+1% and +3%), well within what a different random seed could change. Their stronger regularisation (`min_child_samples`, `min_samples_leaf`) already prevented them from over-fitting the encoding.
* The OOF and in-sample encodings are almost identical (correlation 0.9999, mean difference $162): the leak is small, but XGBoost's deep trees (`max_depth=8`, `min_child_weight=5`) were able to exploit it.
* **XGBoost + OOF (2,061) and LightGBM + in-sample (2,070) are now tied**: a 0.4% difference is not significant. LightGBM stays better on Thanksgiving/Christmas weeks (MAE 2,990 vs 3,077).

**Decision:** the deployed LightGBM model is kept as is. OOF encoding is the methodologically safer choice and should be used for any model that is sensitive to it (XGBoost here), but it does not improve LightGBM.

## 5. Ideas from the 1st place solution, tested one by one

The competition winner (David Thaler, private WMAE ≈ 2,300 vs 3,070 for our submission) used [per-department time-series models](https://github.com/davidthaler/Walmart_competition_code). Four of his ideas are tested here, **each one alone** against our current model (LightGBM + lags, WMAE ≈ 2,068 on the holiday-aware split), then combined:

| # | Idea | What changes |
|---|---|---|
| A | **Drop external features** | He used no `features.csv` variable: remove Temperature, Fuel_Price, MarkDowns, CPI, Unemployment |
| B | **Christmas day-of-week shift** | Move part of each pre-Christmas week's sales to the next week, because Christmas falls on a different weekday every year |
| C | **SVD denoising across stores** | For each department, replace the weeks × stores sales matrix by a low-rank approximation before computing the lag features |
| D | **Ensemble with a time-series model** | Average LightGBM with a seasonal naive forecast built on the SVD-denoised history |

All models use the same split as above (train until 2011-10-28, validate on the next 52 weeks) and the shared code of `walmart_features.py`.

In [17]:
from walmart_features import (load_raw as wf_load_raw, build_base_features as wf_build,
                              add_lag_features as wf_add_lags, make_lgbm_pipeline, FEATURES as WF_FEATURES)
from lightgbm import LGBMRegressor

wf_train, _, wf_store = wf_load_raw()
wf_full = wf_build(wf_train, wf_store)
CUTOFF = pd.Timestamp("2011-10-28")
h_tr = wf_full[wf_full["Date"] <= CUTOFF]
h_va = wf_full[wf_full["Date"] > CUTOFF]

EXTERNAL = ["Temperature", "Fuel_Price", *MARKDOWN_COLS, "CPI", "Unemployment"]


def lgbm_pipeline(feature_cols):
    # same model as make_lgbm_pipeline (CPU), but with a configurable feature list
    numeric = [c for c in feature_cols if c != "Type"]
    pre = ColumnTransformer([("type", OrdinalEncoder(categories=[["C", "B", "A"]]), ["Type"]),
                             ("num", "passthrough", numeric)], verbose_feature_names_out=False)
    model = LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=8, num_leaves=63,
                          subsample=0.8, colsample_bytree=0.8, min_child_samples=20,
                          random_state=42, verbose=-1)
    return Pipeline([("Preprocessing", pre), ("Model", model)])


def fit_predict(train_frame, val_frame, feature_cols):
    pipe = lgbm_pipeline(feature_cols).fit(train_frame[feature_cols], train_frame["Weekly_Sales"])
    return pipe.predict(val_frame[feature_cols])


def score(pred, val_frame=h_va):
    peaks = peak_mask(val_frame["Date"])
    y = val_frame["Weekly_Sales"].values
    return {"WMAE": wmae(y, pred, val_frame["IsHoliday"] == 1),
            "MAE": mean_absolute_error(y, pred),
            "MAE Thanks/Xmas": mean_absolute_error(y[peaks], np.asarray(pred)[peaks])}


tr_ref = wf_add_lags(h_tr, history=h_tr)
va_ref = wf_add_lags(h_va, history=h_tr)
pred_ref = fit_predict(tr_ref, va_ref, WF_FEATURES)

winner_results = [{"Test": "Reference: LightGBM + lags", **score(pred_ref)}]
pd.DataFrame(winner_results).round(0)

,Test,WMAE,MAE,MAE Thanks/Xmas
0,Reference: LightGBM + lags,2068.0,1960.0,2988.0


### A. Drop the external features

In [18]:
FEATURES_NO_EXT = [c for c in WF_FEATURES if c not in EXTERNAL]
pred_a = fit_predict(tr_ref, va_ref, FEATURES_NO_EXT)
winner_results.append({"Test": "A. no external features", **score(pred_a)})
pd.DataFrame(winner_results).round(0)

,Test,WMAE,MAE,MAE Thanks/Xmas
0,Reference: LightGBM + lags,2068.0,1960.0,2988.0
1,A. no external features,2034.0,1931.0,2890.0


### B. Christmas day-of-week shift

Walmart weeks end on a Friday. The number of **shopping days before Christmas** that fall in the Christmas week changes every year:

| Year | Christmas | Christmas week (ending) | Pre-Christmas days in that week |
|---|---|---|---|
| 2010 | Saturday | 2010-12-31 | 0 |
| 2011 | Sunday | 2011-12-30 | 1 (Sat 24) |
| 2012 (test) | Tuesday | 2012-12-28 | 3 (Sat 22 → Mon 24) |

A model that learned from 2010 puts the pre-Christmas rush one day too early when it predicts 2011. Following the winner's post-processing, for every store/department **with a real Christmas peak** (mean of the 3 weeks before Christmas week > 1.1 × mean of the first and last week), each of the 5 weeks ending with Christmas week gives `shift/7` of its predicted sales to the following week. For the validation year the shift is **1 day** (2011 vs 2010); for the 2012 test it would be **2 days** (2012 vs 2011), the value the winner used.

In [19]:
def christmas_shift(val_frame, pred, christmas_week, shift_days, threshold=1.1):
    """Winner's post-processing applied to the 5 weeks ending with Christmas week."""
    weeks = [christmas_week - pd.Timedelta(weeks=k) for k in [4, 3, 2, 1, 0]]
    df = val_frame[["Store", "Dept", "Date"]].copy()
    df["pred"] = pred
    block = (df[df["Date"].isin(weeks)]
             .pivot_table(index=["Store", "Dept"], columns="Date", values="pred")
             .reindex(columns=weeks).fillna(0))
    w = block.values
    peak = w[:, 1:4].mean(axis=1) > threshold * w[:, [0, 4]].mean(axis=1)
    shifted = w.copy()
    shifted[:, 1:] = (1 - shift_days / 7) * w[:, 1:] + (shift_days / 7) * w[:, :-1]
    w = np.where(peak[:, None], shifted, w)
    new = (pd.DataFrame(w, index=block.index, columns=weeks)
           .stack().rename("shifted").reset_index().rename(columns={"level_2": "Date"}))
    out = df.merge(new, on=["Store", "Dept", "Date"], how="left")
    print(f"Christmas shift applied to {peak.sum():,} / {len(peak):,} store/department series")
    return np.where(out["shifted"].notna(), out["shifted"], out["pred"])


XMAS_VAL = pd.Timestamp("2011-12-30")
pred_b = christmas_shift(h_va, pred_ref, XMAS_VAL, shift_days=1)
winner_results.append({"Test": "B. Christmas shift (1 day)", **score(pred_b)})

# sensitivity: is 1 day really the right amount?
for s in [0.5, 1.5, 2]:
    r = score(christmas_shift(h_va, pred_ref, XMAS_VAL, shift_days=s))
    print(f"  shift {s} days -> WMAE {r['WMAE']:,.0f}")
pd.DataFrame(winner_results).round(0)

Christmas shift applied to 2,770 / 3,115 store/department series
Christmas shift applied to 2,770 / 3,115 store/department series
  shift 0.5 days -> WMAE 2,073
Christmas shift applied to 2,770 / 3,115 store/department series
  shift 1.5 days -> WMAE 2,137
Christmas shift applied to 2,770 / 3,115 store/department series
  shift 2 days -> WMAE 2,188


,Test,WMAE,MAE,MAE Thanks/Xmas
0,Reference: LightGBM + lags,2068.0,1960.0,2988.0
1,A. no external features,2034.0,1931.0,2890.0
2,B. Christmas shift (1 day),2098.0,1956.0,3258.0


**Does the shift exist in the data?** Comparing the actual total sales of the 5 weeks ending with Christmas week:

In [20]:
weekly_total = wf_full.groupby("Date")["Weekly_Sales"].sum() / 1e6
for year, xmas in [(2010, pd.Timestamp("2010-12-31")), (2011, XMAS_VAL)]:
    weeks = [xmas - pd.Timedelta(weeks=k) for k in [4, 3, 2, 1, 0]]
    print(year, " | ".join(f"{d.date()}: {weekly_total[d]:.1f}M$" for d in weeks))

2010 2010-12-03: 49.9M$ | 2010-12-10: 55.7M$ | 2010-12-17: 61.8M$ | 2010-12-24: 80.9M$ | 2010-12-31: 40.4M$
2011 2011-12-02: 49.4M$ | 2011-12-09: 55.6M$ | 2011-12-16: 60.1M$ | 2011-12-23: 77.0M$ | 2011-12-30: 46.0M$


The shift is real but **local and smaller than one day**: from 2010 to 2011 about 4–6 M$ move from the week before Christmas to Christmas week (≈ half a day of sales), while the early-December weeks barely change. The winner's rule shifts all 4 transitions. A **targeted** variant only moves sales from the week before Christmas into Christmas week:

In [21]:
def christmas_shift_targeted(val_frame, pred, christmas_week, shift_days):
    df = val_frame[["Store", "Dept", "Date"]].copy()
    df["pred"] = np.asarray(pred, dtype=float)
    before = christmas_week - pd.Timedelta(weeks=1)
    pair = df[df["Date"].isin([before, christmas_week])].pivot_table(
        index=["Store", "Dept"], columns="Date", values="pred").reindex(columns=[before, christmas_week]).fillna(0)
    moved = (shift_days / 7) * pair[before]
    adj = pd.concat([(pair[before] - moved).rename(before), (pair[christmas_week] + moved).rename(christmas_week)], axis=1)
    adj = adj.stack().rename("adj").reset_index().rename(columns={"level_2": "Date"})
    out = df.merge(adj, on=["Store", "Dept", "Date"], how="left")
    return np.where(out["adj"].notna(), out["adj"], out["pred"]).astype(float)


for s in [0.5, 1]:
    pred_b2 = christmas_shift_targeted(h_va, pred_ref, XMAS_VAL, shift_days=s)
    winner_results.append({"Test": f"B2. targeted Christmas shift ({s} day)", **score(pred_b2)})
pd.DataFrame(winner_results).round(0)

,Test,WMAE,MAE,MAE Thanks/Xmas
0,Reference: LightGBM + lags,2068.0,1960.0,2988.0
1,A. no external features,2034.0,1931.0,2890.0
2,B. Christmas shift (1 day),2098.0,1956.0,3258.0
3,B2. targeted Christmas shift (0.5 day),2112.0,1961.0,3346.0
4,B2. targeted Christmas shift (1 day),2222.0,1991.0,4078.0


**Why does the shift hurt?** Total predicted vs actual sales around Christmas 2011:

In [22]:
check = h_va[["Date"]].assign(Actual=h_va["Weekly_Sales"].values, LightGBM=pred_ref,
                              Last_year=va_ref["Sales_Lag_52"].values)
(check.groupby("Date").sum().loc["2011-12-02":"2011-12-30"] / 1e6).round(1)

,Actual,LightGBM,Last_year
Date,,,
2011-12-02,49.4,53.7,49.9
2011-12-09,55.6,58.8,55.7
2011-12-16,60.1,65.7,61.8
2011-12-23,77.0,85.7,80.9
2011-12-30,46.0,46.2,40.4


LightGBM **already predicts Christmas week correctly** (46.2 M$ vs 46.0 M$ actual, while last year's value was 40.4 M$): it learned the effect from `Days_to_Christmas` and `IsHoliday`. Shifting sales into Christmas week therefore double-counts it. Its real December problem is a **level over-estimate** of the pre-Christmas weeks (85.7 M$ vs 77.0 M$ the week before Christmas), which a day-of-week shift cannot fix.

### C. SVD denoising across stores

For each department, the training sales form a **weeks × stores** matrix. Stores share most of the department's seasonality; what is specific to one store is partly noise. Keeping only the first `k` singular components (the winner reduced the rank from 45 to 10–15) gives a denoised history. The **lag features and store/department statistics** are then computed on this denoised history (the target itself stays the real sales).

In [23]:
def svd_denoise(history, k):
    parts = []
    for dept, g in history.groupby("Dept"):
        m = g.pivot_table(index="Date", columns="Store", values="Weekly_Sales")
        observed = m.notna()
        x = m.fillna(0).values
        if min(x.shape) > k:
            u, s, vt = np.linalg.svd(x, full_matrices=False)
            x = (u[:, :k] * s[:k]) @ vt[:k]
        denoised = pd.DataFrame(x, index=m.index, columns=m.columns).where(observed)
        parts.append(denoised.stack().dropna().rename("Weekly_Sales").reset_index().assign(Dept=dept))
    return pd.concat(parts, ignore_index=True)[["Store", "Dept", "Date", "Weekly_Sales"]]


svd_scores = {}
for k in [10, 15]:
    hist_svd = svd_denoise(h_tr, k)
    tr_svd = wf_add_lags(h_tr, history=hist_svd)
    va_svd = wf_add_lags(h_va, history=hist_svd)
    pred_c = fit_predict(tr_svd, va_svd, WF_FEATURES)
    svd_scores[k] = (pred_c, hist_svd)
    winner_results.append({"Test": f"C. SVD-denoised lags (rank {k})", **score(pred_c)})
pd.DataFrame(winner_results).round(0)

,Test,WMAE,MAE,MAE Thanks/Xmas
0,Reference: LightGBM + lags,2068.0,1960.0,2988.0
1,A. no external features,2034.0,1931.0,2890.0
2,B. Christmas shift (1 day),2098.0,1956.0,3258.0
3,B2. targeted Christmas shift (0.5 day),2112.0,1961.0,3346.0
4,B2. targeted Christmas shift (1 day),2222.0,1991.0,4078.0
5,C. SVD-denoised lags (rank 10),2079.0,1971.0,2993.0
6,C. SVD-denoised lags (rank 15),2071.0,1961.0,3006.0


### D. Ensemble with a seasonal naive forecast

The seasonal naive forecast ("same week last year") was the strongest simple benchmark of the competition. Built on the **SVD-denoised** history, it is the winner's simplest model. Its errors are very different from LightGBM's, which is what makes an average useful. Missing lags fall back on the store/department mean.

In [24]:
best_k = min(svd_scores, key=lambda k: score(svd_scores[k][0])["WMAE"])
va_snaive = wf_add_lags(h_va, history=svd_scores[best_k][1])
pred_snaive = va_snaive["Sales_Lag_52"].fillna(va_snaive["Store_Dept_Mean"]).fillna(0).values
print(f"Seasonal naive alone (SVD rank {best_k}): WMAE {score(pred_snaive)['WMAE']:,.0f}")

pred_d = 0.5 * pred_ref + 0.5 * pred_snaive
winner_results.append({"Test": "D. 50/50 LightGBM + seasonal naive", **score(pred_d)})

# weight sensitivity (for information: choosing it on the validation set would be optimistic)
for w in [0.6, 0.7, 0.8, 0.9]:
    print(f"  {w:.0%} LightGBM -> WMAE {score(w * pred_ref + (1 - w) * pred_snaive)['WMAE']:,.0f}")
pd.DataFrame(winner_results).round(0)

Seasonal naive alone (SVD rank 15): WMAE 1,900
  60% LightGBM -> WMAE 1,848
  70% LightGBM -> WMAE 1,887
  80% LightGBM -> WMAE 1,937
  90% LightGBM -> WMAE 1,998


,Test,WMAE,MAE,MAE Thanks/Xmas
0,Reference: LightGBM + lags,2068.0,1960.0,2988.0
1,A. no external features,2034.0,1931.0,2890.0
2,B. Christmas shift (1 day),2098.0,1956.0,3258.0
3,B2. targeted Christmas shift (0.5 day),2112.0,1961.0,3346.0
4,B2. targeted Christmas shift (1 day),2222.0,1991.0,4078.0
5,C. SVD-denoised lags (rank 10),2079.0,1971.0,2993.0
6,C. SVD-denoised lags (rank 15),2071.0,1961.0,3006.0
7,D. 50/50 LightGBM + seasonal naive,1822.0,1769.0,2284.0


In [25]:
# seasonal naive alone, without SVD and for several ranks
for k in [None, 5, 10, 15, 20]:
    hist = h_tr if k is None else svd_denoise(h_tr, k)
    v = wf_add_lags(h_va, history=hist)
    p_naive = v["Sales_Lag_52"].fillna(v["Store_Dept_Mean"]).fillna(0).values
    print(f"Seasonal naive, {'no SVD' if k is None else f'SVD rank {k}'}: WMAE {score(p_naive)['WMAE']:,.0f}")

Seasonal naive, no SVD: WMAE 1,937


Seasonal naive, SVD rank 5: WMAE 1,890


Seasonal naive, SVD rank 10: WMAE 1,887


Seasonal naive, SVD rank 15: WMAE 1,900


Seasonal naive, SVD rank 20: WMAE 1,913


### Combining the ideas that helped

In [26]:
ref_wmae = winner_results[0]["WMAE"]
helped = {r["Test"][0]: r["WMAE"] for r in winner_results[1:] if r["WMAE"] < ref_wmae}
print("Ideas that improved WMAE on their own:", sorted(helped) or "none")

feature_cols = FEATURES_NO_EXT if "A" in helped else WF_FEATURES
history = svd_scores[best_k][1] if "C" in helped else h_tr
tr_comb = wf_add_lags(h_tr, history=history)
va_comb = wf_add_lags(h_va, history=history)
pred_comb = fit_predict(tr_comb, va_comb, feature_cols)
if "D" in helped:
    pred_comb = 0.5 * pred_comb + 0.5 * pred_snaive
if "B" in helped:
    pred_comb = christmas_shift(h_va, pred_comb, XMAS_VAL, shift_days=1)

winner_results.append({"Test": "Combined: " + " + ".join(sorted(helped)), **score(pred_comb)})
winner_table = pd.DataFrame(winner_results)
winner_table["vs reference"] = (winner_table["WMAE"] / ref_wmae - 1) * 100
winner_table.round({"WMAE": 0, "MAE": 0, "MAE Thanks/Xmas": 0, "vs reference": 1})

Ideas that improved WMAE on their own: ['A', 'D']


,Test,WMAE,MAE,MAE Thanks/Xmas,vs reference
0,Reference: LightGBM + lags,2068.0,1960.0,2988.0,0.0
1,A. no external features,2034.0,1931.0,2890.0,-1.6
2,B. Christmas shift (1 day),2098.0,1956.0,3258.0,1.4
3,B2. targeted Christmas shift (0.5 day),2112.0,1961.0,3346.0,2.1
4,B2. targeted Christmas shift (1 day),2222.0,1991.0,4078.0,7.4
5,C. SVD-denoised lags (rank 10),2079.0,1971.0,2993.0,0.5
6,C. SVD-denoised lags (rank 15),2071.0,1961.0,3006.0,0.2
7,D. 50/50 LightGBM + seasonal naive,1822.0,1769.0,2284.0,-11.9
8,Combined: A + D,1835.0,1776.0,2328.0,-11.3


**Reading:**

| Idea | WMAE | vs reference | Verdict |
|---|---|---|---|
| Reference: LightGBM + lags | 2,068 | | |
| A. No external features | 2,034 | -1.6% | ✅ small gain, consistent with the winner ignoring them |
| B. Christmas shift (winner's rule, 1 day) | 2,098 | +1.4% | ❌ |
| B2. Targeted Christmas shift (0.5 / 1 day) | 2,112 / 2,222 | +2.1% / +7.4% | ❌ |
| C. SVD-denoised lags (rank 10 / 15) | 2,079 / 2,071 | +0.5% / +0.2% | ➖ neutral for LightGBM |
| **D. 50/50 LightGBM + SVD seasonal naive** | **1,822** | **-11.9%** | ✅ **best** |
| Combined A + D | 1,835 | -11.3% | ➖ no better than D alone |

* **The ensemble is by far the biggest gain (-11.9%)**, and it also cuts the error on Thanksgiving/Christmas weeks from 2,988 to 2,284. The surprise: **the seasonal naive forecast alone already beats LightGBM**: WMAE 1,937 with the raw history, ~1,890 with SVD denoising (rank 5 to 15 all within 1,887–1,900), vs 2,068 for LightGBM. "Same week last year, denoised across stores" is a very strong forecast for this data, and the two models make different errors, so their average beats both.
* **SVD denoising does not help LightGBM's lag features**, but it improves the seasonal naive forecast by 2–3%: the winner's key idea works for time-series models, not as a feature for a tree model.
* **The Christmas shift hurts here** because LightGBM already captures the 2010 → 2011 day-of-week effect (see above). This validation only tests a 1-day difference; the 2012 test year has a 3-day difference (Christmas on a Tuesday), where the effect is larger and could still matter. It cannot be validated locally.
* **Dropping the external features** gives a small gain on its own, but not on top of the ensemble (1,835 vs 1,822): a difference of this size is within noise.
* The 50/50 weight was fixed in advance on purpose. The weight sensitivity above shows neighbouring weights give similar results, so the gain does not depend on a lucky choice. The SVD rank (15) was picked from the LightGBM test (C); for the seasonal naive forecast every rank from 5 to 15 gives almost the same score, so the result does not depend on it.

**Recommended next step:** make the final model the **50/50 average of LightGBM + lags and the SVD seasonal naive forecast** (validation WMAE 1,822 vs 2,068 today).

## 6. Validation over several periods (backtest)

**Why this step?** All the comparisons above rely on a **single** validation window. A difference of 1% between two models might just be luck of that particular period. To measure how much scores move from one period to another, the final model and its two parts are evaluated on **3 different windows**.

Each window lasts **39 weeks**, the same horizon as the Kaggle test, and the model is always trained only on the data before it:

| Window | Train until | Validate on | Holiday weeks included |
|---|---|---|---|
| A | 2011-07-29 | 2011-08-05 → 2012-04-27 | Labor Day, Thanksgiving, Christmas, Super Bowl |
| B | 2011-10-28 | 2011-11-04 → 2012-07-27 | Thanksgiving, Christmas, Super Bowl (same calendar span as the test) |
| C | 2012-01-27 | 2012-02-03 → 2012-10-26 | Super Bowl, Labor Day (no Thanksgiving / Christmas) |

The 30/70 weighting is included to check whether it is really better than the 50/50 fixed in advance.

In [27]:
from walmart_features import svd_denoise as wf_svd_denoise, predict_ensemble as wf_predict_ensemble

windows = {"A": "2011-07-29", "B": "2011-10-28", "C": "2012-01-27"}
backtest = []
for name, cutoff in windows.items():
    cutoff = pd.Timestamp(cutoff)
    fold_train = wf_full[wf_full["Date"] <= cutoff]
    fold_val = wf_full[(wf_full["Date"] > cutoff) &
                       (wf_full["Date"] <= cutoff + pd.Timedelta(weeks=39))].reset_index(drop=True)

    fold_lag = wf_add_lags(fold_train, history=fold_train)
    fold_model = make_lgbm_pipeline().fit(fold_lag[WF_FEATURES], fold_lag["Weekly_Sales"])
    ens, lgbm_p, naive_p = wf_predict_ensemble(fold_model, fold_val, history=fold_train,
                                               denoised_history=wf_svd_denoise(fold_train))

    y, hol = fold_val["Weekly_Sales"].values, fold_val["IsHoliday"] == 1
    for model_name, pred in [("LightGBM + lags", lgbm_p), ("Seasonal naive (SVD)", naive_p),
                             ("Ensemble 50/50", ens), ("Ensemble 30/70", 0.3 * lgbm_p + 0.7 * naive_p)]:
        backtest.append({"Window": name, "Model": model_name, "WMAE": wmae(y, pred, hol)})

backtest = pd.DataFrame(backtest).pivot(index="Model", columns="Window", values="WMAE")
backtest["Mean"] = backtest[list(windows)].mean(axis=1)
backtest["Std"] = backtest[list(windows)].std(axis=1)
backtest.sort_values("Mean").round(0)

Window,A,B,C,Mean,Std
Model,,,,,
Ensemble 30/70,1853.0,1867.0,1654.0,1791.0,119.0
Ensemble 50/50,1875.0,1889.0,1635.0,1799.0,143.0
Seasonal naive (SVD),1941.0,1960.0,1738.0,1880.0,123.0
LightGBM + lags,2171.0,2183.0,1707.0,2020.0,271.0


In [28]:
gain_vs_lgbm = (1 - backtest.loc["Ensemble 50/50", list(windows)] / backtest.loc["LightGBM + lags", list(windows)]) * 100
gain_vs_naive = (1 - backtest.loc["Ensemble 50/50", list(windows)] / backtest.loc["Seasonal naive (SVD)", list(windows)]) * 100
diff_30_70 = backtest.loc["Ensemble 30/70", list(windows)] - backtest.loc["Ensemble 50/50", list(windows)]
pd.DataFrame({"Ensemble gain vs LightGBM (%)": gain_vs_lgbm, "Ensemble gain vs naive (%)": gain_vs_naive,
              "30/70 minus 50/50 (WMAE)": diff_30_70}).round(1)

,Ensemble gain vs LightGBM (%),Ensemble gain vs naive (%),30/70 minus 50/50 (WMAE)
Window,,,
A,13.6,3.4,-21.5
B,13.5,3.6,-22.1
C,4.2,5.9,19.6


**Reading:**

* **The ensemble beats both of its parts in every window**: by 4 to 14% against LightGBM and by 3 to 6% against the seasonal naive forecast. The gain does not depend on the choice of the validation period.
* **Scores move a lot from one window to another** (standard deviation of about 140 WMAE for the ensemble, close to 8% of its mean). Window C is much easier because it contains neither Thanksgiving nor Christmas. A 1% difference measured on a single period is therefore far below this natural variability, which confirms that such differences should be treated as noise.
* **30/70 is not consistently better than 50/50**: it wins on windows A and B (by 22 WMAE) but loses on C (by 20). Its small advantage on the original validation was luck of that period, so keeping the 50/50 weight fixed in advance was the right decision.
* LightGBM is the least stable part (standard deviation of about 270), and the seasonal naive forecast the most stable. Averaging them gives a model that is both more accurate and more stable than LightGBM alone.

## Conclusions

### 1. Validation split: the old scores were optimistic

| Model | WMAE old split (no Thanksgiving/Christmas) | WMAE new split (with them) |
|---|---|---|
| Random Forest | 2,364 | 2,779 (+18%) |
| XGBoost | 2,752 | 2,919 (+6%) |
| LightGBM | 3,124 | 3,501 (+12%) |

Every model gets worse once the holiday peaks are in the validation window. The new split is the honest benchmark.

### 2. Lag features: the biggest improvement of the project

| Model | WMAE base | WMAE + lags | Gain | MAE on Thanksgiving/Christmas weeks |
|---|---|---|---|---|
| **LightGBM** | 3,501 | **2,070** | **-41%** | 4,605 → **2,989** |
| Random Forest | 2,779 | 2,293 | -17% | 4,070 → 4,425 |
| XGBoost | 2,919 | 2,645 | -9% | 3,747 → 4,137 |

* **LightGBM + lags is the new best model** (WMAE 2,070, R² 0.9637), and the only one that also improves on the Thanksgiving/Christmas weeks.
* The gain comes mostly from **`Store_Dept_Mean` / `Store_Dept_Median`** (~87% of LightGBM's total gain): knowing the usual sales level of each store/department pair is far more informative than `Store` and `Dept` as raw IDs. The 52-week lags add the "same week last year" signal (~5%).
* Only lags of **52 weeks or more** were used, because the Kaggle test set goes up to 39 weeks ahead: shorter lags would not be available at prediction time.
* ⚠️ XGBoost with lags improves its WMAE but its **R² drops from 0.948 to 0.812**: a few very large errors appear. Section 4 shows why: XGBoost over-fits the in-sample target encoding, and out-of-fold encoding fixes it (R² 0.965, WMAE 2,061).

### 3. Retraining on all data

The final LightGBM + lags model was retrained on **2010-02-05 → 2012-10-26** and saved to `walmart_submission_v2.csv` (**not submitted**). It cannot be scored without Kaggle, but a plausibility check against the same season one year earlier is consistent:

| Total weekly sales (M$) | Thanksgiving week | Mean Nov → Jul |
|---|---|---|
| Actual 2011–12 | 66.6 | 48.6 |
| v1 submission (XGBoost, trained 2010–2011) | 64.9 | 46.2 |
| **v2 submission (LightGBM + lags, trained 2010–2012)** | **68.4** | **49.5** |

Sales were stable to slightly growing year-over-year (+2.5% on comparable weeks in 2012), and v2 follows that level while v1, which never saw 2012, sits below it.

### 5. Ideas from the 1st place solution

Averaging LightGBM with an SVD-denoised seasonal naive forecast gives **WMAE 1,822 (-12%)**, the best result of the project. Dropping external features gives a small gain; SVD-denoised lags and the Christmas day-of-week shift do not help LightGBM (see section 5).

### 6. Validation over several periods

On 3 windows of 39 weeks, the 50/50 ensemble beats both of its parts every time (mean WMAE 1,799 vs 2,020 for LightGBM). Scores vary by about 8% from one window to another, so 1% differences on a single period are noise, and the 30/70 weighting is not consistently better than 50/50.

### Next steps
* ✅ Done: LightGBM + lag features is now the final model of `Notebook.ipynb` and the model deployed in the app (feature engineering shared through `walmart_features.py`).
* Tune LightGBM's hyper-parameters on the new split (they were copied from the original notebook).
* ✅ Done: the LightGBM + SVD seasonal naive ensemble (SVD rank 10) is now the final model of `Notebook.ipynb` and the app (validation WMAE 1,818).
* ✅ XGBoost's R² drop with lag features is explained by the in-sample target encoding leak: see section 4, fixed by out-of-fold encoding.